# 01 - Build the Master Patient Table

## LLM-Assisted Patient Similarity

This notebook prepares the synthetic Synthea COVID-19 dataset for the patient-similarity and LLM prompting experiments.

### Goals
- Load the required Synthea tables.
- Inspect the source data.
- Build one patient-level master table.
- Validate the processes dataset.
- Save the processed table for later milestones.

The reusable preprocessing logic lives in `src/preprocessing.py`; this notebook demonstrates the workflow.

## 1. Project Setup

The note book is stored in `notebooks/`, while the reusable Python modules are stored in `src/`.
The project root is therefore added to Python's import path before importing the preprocessing functions.

In [ ]:
from pathlib import Path
import sys

# This notebook is expected to run from the project's notebooks/ directory
NOTEBOOK_DIR = Path.cwd()

if NOTEBOOK_DIR.name == "notebooks":
    PROJECT_ROOT = NOTEBOOK_DIR.parent
else:
    # Helpful fallback when the Jupyter working directory is the project root
    PROJECT_ROOT = NOTEBOOK_DIR

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Notebook directory:", NOTEBOOK_DIR)
print("Project root:", PROJECT_ROOT)
print("src exists:", (PROJECT_ROOT / "src").exists())

## 2. Import Preprocessing Functions

These functions are implemented in `src/preprocessing.py`

In [ ]:
from src.preprocessing import (
    load_synthea_data,
    calculate_age,
    build_patient_master,
    save_patient_master
)

## 3. Define Data Paths

Update `RAW_DATA_DIR` only if your extracted Synthea folder has a different name or location

In [ ]:
RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw" / "10k_synthea_covid19_csv"
PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"

PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

print("Raw data:", RAW_DATA_DIR)
print("Processed data:", PROCESSED_DATA_DIR)
print("Raw data exists:", RAW_DATA_DIR.exists())

## 4. Load the Synthea Tables

* `patients.csv` - demographics and healthcare costs
* `conditions.csv` - diagnoses and clinical condtions
* `encounters.csv` - healthcare encounters
* `medications.csv` - medication history

In [ ]:
patients, conditions, encounters, medications = load_synthea_data(RAW_DATA_DIR)

print("Patients:", patients.shape)
print("Conditions:", conditions.shape)
print("Encounters:", encounters.shape)
print("Medications:", medications.shape)

## 5. Inspect the Raw Data

In [ ]:
patients.head()

In [ ]:
conditions.head()

In [ ]:
encounters.head()

In [ ]:
medications.head()

## Calculate Patient Age

Age is derived from `BIRTHDATE` by the reusable `calculate_age` function

In [ ]:
patients = calculate_age(patients)

patients[["Id", "BIRTHDATE", "AGE", "GENDER", "RACE", "ETHNICITY"]].head()

## Build the Patient Master Table

The master table aggregates the source tables so that each row represents one patient.
It contains demographics, healthcare costs, a list of unique conditions, a list of unique medications, and encounter count.

In [ ]:
patient_df = build_patient_master(
    patients,
    conditions,
    medications,
    encounters,
)

print("Patients master shape:", patient_df.shape)
patient_df.head()

## 8. Validate the Master Table

In [ ]:
print("Number of rows:", len(patient_df))
print("Unique patients:", patient_df["PATIENT"].nunique)
print("Duplicate patient IDs:", patient_df["PATIENT"].duplicated().sum())
print()
print("Columns:")
print(patient_df.columns.tolist())

In [ ]:
print("CONDITIONS type:", type(patient_df.loc[0, "CONDITIONS"]))
print("MEDICATIONS type:", type(patient_df.loc[0, "MEDICATIONS"]))

print("\nExample conditions:")
print(patient_df.loc[0, "CONDITIONS"])

print("\nExample medications:")
print(patient_df.loc[0, "MEDICATIONS"])

## 9. Basic Quality Checks

In [ ]:
patient_df[
    [
        "AGE",
        "HEALTHCARE_EXPENSES",
        "HEALTHCARE_COVERAGE",
        "ENCOUNTER_COUNT"
    ]
].describe()

In [ ]:
patient_df[
    [
        "PATIENT",
        "AGE",
        "GENDER",
        "RACE",
        "ETHNICITY",
        "CONDITIONS",
        "MEDICATIONS",
        "ENCOUNTER_COUNT",
    ]
].isna().sum()

## 10. Save the Processed Dataset

The preprocessing module saves the patient master table for reuse in later notebooks.

The pickle version preserves Python list columns. A CSV version is also kept as a human-readable artifact.

In [ ]:
save_patient_master(
    patient_df,
    PROCESSED_DATA_DIR
)

print("Patient master dataset saved successfully.")

## 11. Verify the Saved Pickle

In [ ]:
import pandas as pd

saved_file = PROCESSED_DATA_DIR / "patient_master.pkl"

reloaded_df = pd.read_pickle(saved_file)

print("Reloaded shape:", reloaded_df.shape)
print("CONDITIONS type after reload:", type(reloaded_df.loc[0, "CONDITIONS"]))
print("MEDICATIONS type after reload:", type(reloaded_df.loc[0, "MEDICATIONS"]))

reloaded_df.head()

## Milestone 1 Complete

The raw Synthea tables have now been transformed into a reusable patient-level dataset.